### Main Exercise - Normalizing Actor-Genre Matrix

In [24]:
import json
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.metrics import DistanceMetric
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
from pathlib import Path

folder_path = Path.home() / "Downloads" / "INST414"

matches = list(
    folder_path.rglob("imdb_movies_2000to2022.prolific*.json")
)

if not matches:
    raise FileNotFoundError(
        f"No IMDb JSON file found inside: {folder_path}"
    )

file_path = matches[0]

print("Found file:", file_path)

Found file: /Users/rgaffney/Downloads/INST414/imdb_movies_2000to2022.prolific.json


#### Build the actor-by-genre counts

In [26]:
# Stores genre counts for each actor
actor_genre_counts = defaultdict(lambda: defaultdict(int))

# Stores actor names
actor_id_to_name = {}

with open(file_path, 'r', encoding='utf-8') as in_file:
    for line in in_file:
        line = line.strip()

        if not line:
            continue

        movie = json.loads(line)

        genres = movie.get('genres', [])
        actors = movie.get('actors', [])

        for actor_id, actor_name in actors: actor_id_to_name[actor_id] = actor_name
        for genre in genres:
            actor_genre_counts[actor_id][genre] += 1

print("Number of actors:", len(actor_genre_counts))


Number of actors: 13787


##### Creating the feature matrix

In [27]:
actor_genre_df = pd.DataFrame.from_dict(actor_genre_counts, orient='index').fillna(0)

# Replace missing genre appearence with 0
actor_genre_df = actor_genre_df.fillna(0).astype(int)
actor_genre_df.index.name = 'actor_id'
print("Feature matrix shape:", actor_genre_df.shape)
actor_genre_df.head()

Feature matrix shape: (13787, 25)


,Comedy,Fantasy,Romance,Adventure,Family,Horror,Sci-Fi,Drama,Action,Crime,...,History,,Music,Sport,News,Western,Musical,War,Short,Reality-TV
actor_id,,,,,,,,,,,,,,,,,,,,,
nm0005227,2,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nm0329491,1,0,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nm0412917,2,0,1,1,0,0,0,1,4,1,...,0,0,0,0,0,0,0,0,0,0
nm0803138,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nm0732133,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


##### Applying L1 Normalization

In [28]:
# Calculate the sum of every's actor's row
row_sums = actor_genre_df.sum(axis=1)

# Create a new L1-normalized feature matrix
actor_genre_normalized_df = actor_genre_df.div(row_sums, axis=0)
actor_genre_normalized_df.head()

,Comedy,Fantasy,Romance,Adventure,Family,Horror,Sci-Fi,Drama,Action,Crime,...,History,,Music,Sport,News,Western,Musical,War,Short,Reality-TV
actor_id,,,,,,,,,,,,,,,,,,,,,
nm0005227,0.333333,0.166667,0.166667,0.166667,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0329491,0.333333,0.000000,0.000000,0.000000,0.000000,0.333333,0.333333,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0412917,0.181818,0.000000,0.090909,0.090909,0.000000,0.000000,0.000000,0.090909,0.363636,0.090909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0803138,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0732133,0.333333,0.000000,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Verifying every row sums to 1

In [41]:
normalized_row_sums = actor_genre_normalized_df.sum(axis=1)
print(normalized_row_sums.head())
print("Do all rows sum to 1?", np.allclose(normalized_row_sums, 1))

actor_id
nm0005227    1.0
nm0329491    1.0
nm0412917    1.0
nm0803138    1.0
nm0732133    1.0
dtype: float64
Do all rows sum to 1? True


#### Selecting Chris Hemsworth

In [ ]:
query_actor_id = "nm1165110" 
if query_actor_id not in actor_genre_normalized_df.index:
    raise ValueError(f"Actor ID {query_actor_id} not found in the feature dataset.")

query_actor_name = actor_id_to_name.get(query_actor_id, "Unknown")
print("Query actor:", query_actor_id, query_actor_name)

query_vector = actor_genre_normalized_df.loc[[query_actor_id]].to_numpy(dtype=float)

Query actor: nm1165110 Chris Hemsworth


#### Calculating Euclidean Distances

In [43]:
euclidean_metric = DistanceMetric.get_metric('euclidean')
normalized_feature_matrix = actor_genre_l1_df.to_numpy(dtype=float)
distances = euclidean_metric.pairwise(query_vector, normalized_feature_matrix).flatten()
print("Number of distances calculated:", len(distances))

Number of distances calculated: 13787


#### Create the Euclidean Results

In [48]:
euclidean_results_df = pd.DataFrame({"actor_id": actor_genre_l1_df.index, 
    "actor_name": [actor_id_to_name.get(actor_id, "Unknown") for actor_id in 
                   actor_genre_l1_df.index], "euclidean_distance": distances})

# Remove Chris Hemsworth because hsi distance from himself is 0
euclidean_results_df = euclidean_results_df[euclidean_results_df['actor_id'] != query_actor_id]

top_10_euclidean = (
    euclidean_results_df
    .sort_values('euclidean_distance')
    .head(10)
    .reset_index(drop=True)
)
top_10_euclidean

,actor_id,actor_name,euclidean_distance
0,nm0000602,Robert Redford,0.000000
1,nm0000178,Diane Lane,0.000000
2,nm0000295,Kate Beckinsale,0.166667
3,nm0068260,Jamie Bell,0.166667
4,nm2257218,Shahana Goswami,0.166667
5,nm0879085,Tyrese Gibson,0.179173
6,nm0290556,James Franco,0.195434
7,nm0913822,Ken Watanabe,0.207870
8,nm0492373,Phyllida Law,0.235702
9,nm4121613,Bailee Michelle Johnson,0.235702


In [50]:
print ("Top 10 actors most similar to Chris Hemsworth " "using L1-normalized Euclidean distance:\n")
for rank, row in top_10_euclidean.iterrows():
    print(f"{rank + 1}. {row['actor_id']} (ID: {row['actor_name']}) - Distance: {row['euclidean_distance']:.4f}")

Top 10 actors most similar to Chris Hemsworth using L1-normalized Euclidean distance:

1. nm0000602 (ID: Robert Redford) - Distance: 0.0000
2. nm0000178 (ID: Diane Lane) - Distance: 0.0000
3. nm0000295 (ID: Kate Beckinsale) - Distance: 0.1667
4. nm0068260 (ID: Jamie Bell) - Distance: 0.1667
5. nm2257218 (ID: Shahana Goswami) - Distance: 0.1667
6. nm0879085 (ID: Tyrese Gibson) - Distance: 0.1792
7. nm0290556 (ID: James Franco) - Distance: 0.1954
8. nm0913822 (ID: Ken Watanabe) - Distance: 0.2079
9. nm0492373 (ID: Phyllida Law) - Distance: 0.2357
10. nm4121613 (ID: Bailee Michelle Johnson) - Distance: 0.2357


## Comparing the results with cosine similarity

#### Calculate cosine similarity

In [51]:
cosine_scores = cosine_similarity(
    actor_genre_df.to_numpy(dtype=float),
    actor_genre_df.loc[
        [query_actor_id]
    ].to_numpy(dtype=float)
).flatten()

#### Get the cosine top ten

In [52]:
cosine_results_df = pd.DataFrame({
    "actor_id": actor_genre_df.index,
    "actor_name": [
        actor_id_to_name.get(actor_id, "Unknown")
        for actor_id in actor_genre_df.index
    ],
    "cosine_similarity": cosine_scores
})

# Remove Chris Hemsworth
cosine_results_df = cosine_results_df[
    cosine_results_df["actor_id"] != query_actor_id
]

# Higher cosine similarity means more similar
top_10_cosine = (
    cosine_results_df
    .sort_values(
        "cosine_similarity",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

top_10_cosine

,actor_id,actor_name,cosine_similarity
0,nm0000602,Robert Redford,1.000000
1,nm0000178,Diane Lane,1.000000
2,nm2257218,Shahana Goswami,0.948683
3,nm0000295,Kate Beckinsale,0.948683
4,nm0068260,Jamie Bell,0.948683
5,nm0879085,Tyrese Gibson,0.941357
6,nm0290556,James Franco,0.929670
7,nm0913822,Ken Watanabe,0.923133
8,nm0933310,Don Wilson,0.912871
9,nm5875121,Chandini Sreedharan,0.912871


#### Displaying both lists side by side

In [53]:
comparison_df = pd.DataFrame({
    "rank": range(1, 11),
    "L1 Euclidean actor": top_10_euclidean["actor_name"],
    "Euclidean distance": top_10_euclidean[
        "euclidean_distance"
    ],
    "Cosine actor": top_10_cosine["actor_name"],
    "Cosine similarity": top_10_cosine[
        "cosine_similarity"
    ]
})

comparison_df

,rank,L1 Euclidean actor,Euclidean distance,Cosine actor,Cosine similarity
0,1,Robert Redford,0.000000,Robert Redford,1.000000
1,2,Diane Lane,0.000000,Diane Lane,1.000000
2,3,Kate Beckinsale,0.166667,Shahana Goswami,0.948683
3,4,Jamie Bell,0.166667,Kate Beckinsale,0.948683
4,5,Shahana Goswami,0.166667,Jamie Bell,0.948683
5,6,Tyrese Gibson,0.179173,Tyrese Gibson,0.941357
6,7,James Franco,0.195434,James Franco,0.929670
7,8,Ken Watanabe,0.207870,Ken Watanabe,0.923133
8,9,Phyllida Law,0.235702,Don Wilson,0.912871
9,10,Bailee Michelle Johnson,0.235702,Chandini Sreedharan,0.912871


#### Count how many actors appear in both lists

In [54]:
euclidean_ids = set(top_10_euclidean["actor_id"])
cosine_ids = set(top_10_cosine["actor_id"])

common_ids = euclidean_ids.intersection(cosine_ids)

only_euclidean_ids = euclidean_ids.difference(cosine_ids)
only_cosine_ids = cosine_ids.difference(euclidean_ids)

print("Actors appearing in both lists:", len(common_ids))

print("\nActors in both lists:")
for actor_id in common_ids:
    print(actor_id, actor_id_to_name.get(actor_id, "Unknown"))

print("\nOnly in L1 Euclidean list:")
for actor_id in only_euclidean_ids:
    print(actor_id, actor_id_to_name.get(actor_id, "Unknown"))

print("\nOnly in cosine list:")
for actor_id in only_cosine_ids:
    print(actor_id, actor_id_to_name.get(actor_id, "Unknown"))

Actors appearing in both lists: 8

Actors in both lists:
nm0879085 Tyrese Gibson
nm0000602 Robert Redford
nm0068260 Jamie Bell
nm0913822 Ken Watanabe
nm2257218 Shahana Goswami
nm0290556 James Franco
nm0000295 Kate Beckinsale
nm0000178 Diane Lane

Only in L1 Euclidean list:
nm4121613 Bailee Michelle Johnson
nm0492373 Phyllida Law

Only in cosine list:
nm0933310 Don Wilson
nm5875121 Chandini Sreedharan


### Describe how this list has changed compared to Cosine Similarity

##### The L1-normalized Euclidean-distance results were very similar to the cosine similarity results. Eight of the ten actors appeared in both lists: Tyrese Gibson, Robert Redford, Jamie Bell, Ken Watanabe, Shahana Goswami, James Franco, Kate Beckinsale, and Diane Lane. Bailee Michelle Johnson and Phyllida Law appeared only in the L1 Euclidean list, while Don Wilson and Chandini Sreedharan appeared only in the cosine-similarity list.

##### L1 normalization converted each actor's genre counts into proportions that sum to 1, reducing the influence of how many total roles an actor has had. Euclidean distance then measured the absoulute differences between these normalized genre proportions. Cosine similarity instead measured the angle between the actors' genre vectors and was already unaffected by their overall magnitude. Because the two methods measure similarity differently, two actors were replaced and some rankings likely changed. However, the eight-actor overlap indicates that both methods indiftified largely similar genre profiles for Chris Hemsworth.

### Additional Exercise - Adding Movie-Vote Counts to Actor-Genre Matrix